# Chapter 7: Cape Cod and Frequency–Severity Methods

Code of Chapter 7 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 7.1.3 Worked example

In [ ]:
import numpy as np
import pandas as pd
from gicapecod import used_up_base, cape_cod_ratio, cape_cod, average_severity
from gidevelop import average_factors, cumulative_factors, develop

T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=pd.Index([2022, 2023, 2024, 2025], name="accident_year"),
                 columns=pd.Index([12, 24, 36, 48], name="age"))
EP = pd.Series([2500, 2700, 2900, 3100.0], index=T.index)
dev = develop(T, cumulative_factors(average_factors(T, "volume"), 1.0))
used_up_base(EP, dev["cdf"]).sum().round(2)        # 9527.98
cape_cod_ratio(dev["latest"], EP, dev["cdf"])      # 0.6927
cape_cod(dev["latest"], EP, dev["cdf"])["bf_ultimate"].sum().round(2)   # 7758.2

## 7.2.1 Cape Cod pure premium

In [ ]:
from gitriangle import synthetic_history, triangle, exposure_by_accident_year
pol, cl, txs = synthetic_history(2020, 2025, seed=100)
I = triangle(cl, txs, "incurred", 2021, 2025)
P = triangle(cl, txs, "paid", 2021, 2025)
R = triangle(cl, txs, "reported_count", 2021, 2025)
E = exposure_by_accident_year(pol, range(2021, 2026))
dI = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))
dP = develop(P, cumulative_factors(average_factors(P, "volume"), 1.0))
level = pd.Series({y: 1.04 ** -(2025 - y) for y in range(2021, 2026)})
cI = cape_cod(dI["latest"], E["earned_exposure"], dI["cdf"], level)
cI["ratio"].iloc[0].round(2)                       # 491.78

## 7.2.2 Results

In [ ]:
cP = cape_cod(dP["latest"], E["earned_exposure"], dP["cdf"], level)
cI["bf_ultimate"].sum().round(0), cP["bf_ultimate"].sum().round(0)   # 12934671, 12719657

## 7.3.2 Projecting claim counts

In [ ]:
dR = develop(R, cumulative_factors(average_factors(R, "volume"), 1.0))
dR["ultimate"].round(2).tolist()                   # [524.0, 548.0, 580.0, 590.0, 644.45]
(dR["ultimate"] / E["earned_exposure"]).round(4).tolist()

## 7.3.3 Projecting severities by development

In [ ]:
S = average_severity(I, R)
dS = develop(S, cumulative_factors(average_factors(S, "simple"), 1.0))
(dR["ultimate"] * dS["ultimate"]).round(0).tolist()
# [2325132, 2131796, 2351589, 2936973, 3213829]

## Exercise 1: Cape Cod with rate changes

In [ ]:
lvl = pd.Series([1.05 ** 2, 1.05 ** 2, 1.05, 1.0], index=T.index)
cc2 = cape_cod(dev["latest"], EP, dev["cdf"], lvl)
cc2["ratio"].iloc[0].round(4), cc2["bf_ultimate"].sum().round(2)   # 0.6482, 7699.72

## 7.7 Python Walkthrough

In [ ]:
from gicapecod import used_up_base, cape_cod_ratio, cape_cod, average_severity

# Section 7.1: Cape Cod on the example triangle
cc = cape_cod(dev["latest"], EP, dev["cdf"])
cc[["expected_ultimate", "bf_ibnr", "bf_ultimate"]].round(2)

# Section 7.2: synthetic portfolio, exposure deflated by the 4% trend
(E["earned_exposure"] * level / dI["cdf"]).sum().round(2)    # 25521.55 used-up car-years
cP["ratio"].iloc[0].round(2)                                  # 483.61 (paid)

# Section 7.3: frequency-severity
dS["ultimate"].round(2).tolist()           # [4437.27, 3890.14, 4054.46, 4977.92, 4986.91]